In [1]:
from pathlib import Path
import pandas as pd 

In [2]:
# Loading all datasets
folder = Path("../Olist Project/Data/Raw")
files = list(folder.glob("*.csv"))


# Making table summary of datasets profiling
summary = []

# Making missing report by specific column
missing_report_by_columns = {}



for file in files:
    df = pd.read_csv(file)
    # print(file.name,df.shape)
    

    # checking info 
    print(f"\n{'-'*60}")
    print(f"Info: {file.name}")
    df.info()
    
        
    # adding data to summary table           
    summary.append({
        "File" : file.name,
        "Rows" : df.shape[0],
        "Columns" : df.shape[1],
        "Missing values" : df.isnull().sum().sum(),
        "Duplicate Rows" : df.duplicated().sum() })


    # finding missing colunm 
    missing_by_column = df.isna().sum()
    missing_by_column = missing_by_column[missing_by_column > 0]

    # removing non-empty column & printing only empty column
    if not missing_by_column.empty:
        #print(f"{file.name}","\n", missing_by_column,"\n")
        missing_report_by_columns[file.name] = missing_by_column


    
summary_df = pd.DataFrame(summary)


------------------------------------------------------------
Info: olist_customers_dataset.csv
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB

------------------------------------------------------------
Info: olist_geolocation_dataset.csv
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolo

In [3]:
# printing summary tables
summary_df

,File,Rows,Columns,Missing values,Duplicate Rows
0,olist_customers_dataset.csv,99441,5,0,0
1,olist_geolocation_dataset.csv,1000163,5,0,261831
2,olist_orders_dataset.csv,99441,8,4908,0
3,olist_order_items_dataset.csv,112650,7,0,0
4,olist_order_payments_dataset.csv,103886,5,0,0
5,olist_order_reviews_dataset.csv,99224,7,145903,0
6,olist_products_dataset.csv,32951,9,2448,0
7,olist_sellers_dataset.csv,3095,4,0,0
8,product_category_name_translation.csv,71,2,0,0


In [4]:
# printing missing values by column
for file, missing in missing_report_by_columns.items():
    print(f"\n{'-' *20}")
    print(f"Missing values : {file}")
    print(missing)


--------------------
Missing values : olist_orders_dataset.csv
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--------------------
Missing values : olist_order_reviews_dataset.csv
review_comment_title      87656
review_comment_message    58247
dtype: int64

--------------------
Missing values : olist_products_dataset.csv
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [5]:
for file in files:
    df = pd.read_csv(file)

    # checking percentage of missing values by columns
    missing_percentage = (df.isna().sum()/len(df)*100)
    missing_percentage = missing_percentage [ missing_percentage > 0]

    # keeping only missing values columns
    if not missing_percentage.empty:
        print(f"\n{'-'*20}")
        print(f"\n{file.name}")
        print(missing_percentage)



--------------------

olist_orders_dataset.csv
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
dtype: float64

--------------------

olist_order_reviews_dataset.csv
review_comment_title      88.341530
review_comment_message    58.702532
dtype: float64

--------------------

olist_products_dataset.csv
product_category_name         1.851234
product_name_lenght           1.851234
product_description_lenght    1.851234
product_photos_qty            1.851234
product_weight_g              0.006070
product_length_cm             0.006070
product_height_cm             0.006070
product_width_cm              0.006070
dtype: float64


In [6]:
# Data Auditing expected dtypes
expected_dtypes = {
    "customer_zip_code_prefix" : "string",
    "geolocation_zip_code_prefix" : "string",
    "seller_zip_code_prefix" : "string",
    "order_purchase_timestamp" : "datetime",
    "order_approved_at" : "datetime",
    "order_delivered_carrier_date" : "datetime",
    "order_delivered_customer_date" : "datetime",
    "order_estimated_delivery_date" : "datetime",
    "shipping_limit_date" : "datetime",
    "review_creation_date" : "datetime",
    "review_answer_timestamp" : "datetime",
    "product_name_lenght" : "Int64",
    "product_description_lenght" : "Int64",
    "product_photos_qty" : "Int64"
}


# creating Audit tables
audit = []
for file in files:
    df = pd.read_csv(file)

    for column in df.columns:
        current_dtype = str(df[column].dtype)

        if column in expected_dtypes:
            expected_dtype = expected_dtypes[column]
        else:
            expected_dtype = current_dtype
            
        # Taking action according to expected dtypes
        action = "Keep" if current_dtype == expected_dtype else "Convert"

        audit.append ({
        "Dataset" : file.name,
        "Column" : column,
        "Current dtype" : current_dtype,
        "Expected dtype" : expected_dtype,
        "Action" : action
        })

data_audit = pd.DataFrame(audit)
data_audit

,Dataset,Column,Current dtype,Expected dtype,Action
0,olist_customers_dataset.csv,customer_id,str,str,Keep
1,olist_customers_dataset.csv,customer_unique_id,str,str,Keep
2,olist_customers_dataset.csv,customer_zip_code_prefix,int64,string,Convert
3,olist_customers_dataset.csv,customer_city,str,str,Keep
4,olist_customers_dataset.csv,customer_state,str,str,Keep
5,olist_geolocation_dataset.csv,geolocation_zip_code_prefix,int64,string,Convert
6,olist_geolocation_dataset.csv,geolocation_lat,float64,float64,Keep
7,olist_geolocation_dataset.csv,geolocation_lng,float64,float64,Keep
8,olist_geolocation_dataset.csv,geolocation_city,str,str,Keep
9,olist_geolocation_dataset.csv,geolocation_state,str,str,Keep


In [7]:
# To see action for actual conversion of dtype need
data_audit[data_audit["Action"] == "Convert"]

,Dataset,Column,Current dtype,Expected dtype,Action
2,olist_customers_dataset.csv,customer_zip_code_prefix,int64,string,Convert
5,olist_geolocation_dataset.csv,geolocation_zip_code_prefix,int64,string,Convert
13,olist_orders_dataset.csv,order_purchase_timestamp,str,datetime,Convert
14,olist_orders_dataset.csv,order_approved_at,str,datetime,Convert
15,olist_orders_dataset.csv,order_delivered_carrier_date,str,datetime,Convert
16,olist_orders_dataset.csv,order_delivered_customer_date,str,datetime,Convert
17,olist_orders_dataset.csv,order_estimated_delivery_date,str,datetime,Convert
22,olist_order_items_dataset.csv,shipping_limit_date,str,datetime,Convert
35,olist_order_reviews_dataset.csv,review_creation_date,str,datetime,Convert
36,olist_order_reviews_dataset.csv,review_answer_timestamp,str,datetime,Convert


In [8]:
# Starting Actual conversion of dtype { NOTE: if you save again after changing dtype in csv formate then it will not preserved it so you can use .parquet}
cleaned_folder = Path("../Olist Project/Data/Cleaned")

# In case if cleaned folder not available
cleaned_folder.mkdir(parents = True, exist_ok = True)

for file in files:
    df = pd.read_csv(file)

    # Converting ZIP code to string
    zip_cols = [ 
    "customer_zip_code_prefix",
    "geolocation_zip_code_prefix",
    "seller_zip_code_prefix" 
    ]

    for col in zip_cols:
        if col in df.columns:
            df[col] = (df[col].astype("string").str.zfill(5))


    # Converting string to datetime 
    datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp" 
    ]

    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    
    # Converting float to integer
    integer_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
    ]

    for col in integer_cols:
        if col in df.columns:
            df[col] = df[col].astype("Int64")


    # Saving cleaned file 
    output_file = cleaned_folder/file.name
    #optional
    df.to_csv(output_file, index=False)
    
    # Recommended
    parquet_file = cleaned_folder/f"{file.stem}.parquet"
    df.to_parquet(parquet_file, index=False) # To read use this pd.read_parquet("file name")

    # For confirmation
    print(f"Saved:{output_file.name}")
    print(f"Saved:{parquet_file.name}")
    
    

Saved:olist_customers_dataset.csv
Saved:olist_customers_dataset.parquet
Saved:olist_geolocation_dataset.csv
Saved:olist_geolocation_dataset.parquet
Saved:olist_orders_dataset.csv
Saved:olist_orders_dataset.parquet
Saved:olist_order_items_dataset.csv
Saved:olist_order_items_dataset.parquet
Saved:olist_order_payments_dataset.csv
Saved:olist_order_payments_dataset.parquet
Saved:olist_order_reviews_dataset.csv
Saved:olist_order_reviews_dataset.parquet
Saved:olist_products_dataset.csv
Saved:olist_products_dataset.parquet
Saved:olist_sellers_dataset.csv
Saved:olist_sellers_dataset.parquet
Saved:product_category_name_translation.csv
Saved:product_category_name_translation.parquet


In [9]:
''' 
# Showing how parquet file retain schema or dtype
parquet_folder = Path("../Olist Project/Data/Cleaned")
parquet_file = parquet_folder.glob("*.parquet")
for p_file in parquet_file:
    df_parquet = pd.read_parquet(p_file)
    print(f"{'-'*50}")
    df_parquet.info()

# Showing csv not retain schema or dtype after saving so for this we can use other methods
cleaned_csv_folder = Path("../Olist Project/Data/Cleaned")
csv_file = cleaned_csv_folder.glob("*.csv")
for c_file in csv_file:
    df_csv = pd.read_csv(c_file)
    print(f"{'-'*50}")
    df_csv.info
    
'''

' \n# Showing how parquet file retain schema or dtype\nparquet_folder = Path("../Olist Project/Data/Cleaned")\nparquet_file = parquet_folder.glob("*.parquet")\nfor p_file in parquet_file:\n    df_parquet = pd.read_parquet(p_file)\n    print(f"{\'-\'*50}")\n    df_parquet.info()\n\n# Showing csv not retain schema or dtype after saving so for this we can use other methods\ncleaned_csv_folder = Path("../Olist Project/Data/Cleaned")\ncsv_file = cleaned_csv_folder.glob("*.csv")\nfor c_file in csv_file:\n    df_csv = pd.read_csv(c_file)\n    print(f"{\'-\'*50}")\n    df_csv.info\n\n'

In [10]:
# Verification for changed dtype
parquet_folder = Path("../Olist Project/Data/Cleaned")
parquet_file = parquet_folder.glob("*.parquet")
for p_file in parquet_file:
    df_parquet = pd.read_parquet(p_file)
    print(f"{'-'*50}")
    print(f"File: {p_file.name}")
    print(df_parquet.dtypes)


--------------------------------------------------
File: olist_customers_dataset.parquet
customer_id                    str
customer_unique_id             str
customer_zip_code_prefix    string
customer_city                  str
customer_state                 str
dtype: object
--------------------------------------------------
File: olist_geolocation_dataset.parquet
geolocation_zip_code_prefix     string
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object
--------------------------------------------------
File: olist_orders_dataset.parquet
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]


In [11]:
# Another Method for csv files ( Recommended )
cleaned_folder = Path("../Olist Project/Data/Cleaned")
cleaned_files = list(cleaned_folder.glob("*.csv"))

dtype_cols = { "customer_zip_code_prefix" : "string","geolocation_zip_code_prefix" : "string","seller_zip_code_prefix" : "string",
                  "product_name_lenght" : "Int64","product_description_lenght" : "Int64","product_photos_qty" : "Int64"}

date_cols = [ "order_purchase_timestamp","order_approved_at","order_delivered_carrier_date", "order_delivered_customer_date",
                        "order_estimated_delivery_date","shipping_limit_date", "review_creation_date", "review_answer_timestamp"]

for file in cleaned_files:

    # read columns names
    columns = pd.read_csv(file,nrows=0).columns

    # Select only that date columns which are currently present in current csv file
    available_date_cols = [ col for col in date_cols if col in columns ]
    
    df = pd.read_csv(file, dtype = dtype_cols,parse_dates = available_date_cols)
    print(f"{'-'*50}")
    print(file.name)
    print(df.dtypes)
                                

--------------------------------------------------
olist_customers_dataset.csv
customer_id                    str
customer_unique_id             str
customer_zip_code_prefix    string
customer_city                  str
customer_state                 str
dtype: object
--------------------------------------------------
olist_geolocation_dataset.csv
geolocation_zip_code_prefix     string
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object
--------------------------------------------------
olist_orders_dataset.csv
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date 

###### Finding Reason Behind the null values

In [12]:
orders = pd.read_parquet(cleaned_folder/"olist_orders_dataset.parquet")
orders[orders["order_approved_at"].isna()][[
    "order_id",                                                                  
    "order_status",                              
    "order_purchase_timestamp",       
    "order_approved_at",              
    "order_delivered_carrier_date",   
    "order_delivered_customer_date"]]


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
1130,00b1cb0320190ca0daa2c88b35206009,canceled,2018-08-28 15:26:39,NaT,NaT,NaT
1801,ed3efbd3a87bea76c2812c66a0b32219,canceled,2018-09-20 13:54:16,NaT,NaT,NaT
1868,df8282afe61008dc26c6c31011474d02,canceled,2017-03-04 12:14:30,NaT,NaT,NaT
2029,8d4c637f1accf7a88a4555f02741e606,canceled,2018-08-29 16:27:49,NaT,NaT,NaT
2161,7a9d4c7f9b068337875b95465330f2fc,canceled,2017-05-01 16:12:39,NaT,NaT,NaT
...,...,...,...,...,...,...
97696,5a00b4d35edffc56b825c3646a99ba9d,canceled,2017-07-02 15:38:46,NaT,NaT,NaT
98415,227c804e2a44760671a6a5697ea549e4,canceled,2017-09-28 15:02:56,NaT,NaT,NaT
98909,e49e7ce1471b4693482d40c2bd3ad196,canceled,2018-08-07 11:16:28,NaT,NaT,NaT
99283,3a3cddda5a7c27851bd96c3313412840,canceled,2018-08-31 16:13:44,NaT,NaT,NaT


In [13]:
'''
pd.set_option("display.max_rows",200) # set krne ke liye no. of entry show krne ke liye
pd.reset_option("display.max_rows")   # reset krne ke liye
pd.get_option("display.max_rows")     # dekhne ke liye ki kitna pr set hai

# Ye temporary display ke liye
with 
pd.option_context("display.max_rows",200):
    display(variable_name)
'''

'\npd.set_option("display.max_rows",200) # set krne ke liye no. of entry show krne ke liye\npd.reset_option("display.max_rows")   # reset krne ke liye\npd.get_option("display.max_rows")     # dekhne ke liye ki kitna pr set hai\n\n# Ye temporary display ke liye\nwith \npd.option_context("display.max_rows",200):\n    display(variable_name)\n'

In [14]:
# Analyze order statuses for orders with missing approval timestamps
orders.loc[orders["order_approved_at"].isna(), "order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [15]:
# Investigate delivered orders with missing approval timestamps
delivered_missing_approval = orders.loc[ 
    orders["order_approved_at"].isna() & (orders["order_status"] == "delivered")]

delivered_missing_approval

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


In [16]:
# check payment records for delivered orders with missing approval timestamps
payments = pd.read_parquet(cleaned_folder/"olist_order_payments_dataset.parquet")
payment_check = payments[payments["order_id"].isin(delivered_missing_approval["order_id"])]
payment_check

,order_id,payment_sequential,payment_type,payment_installments,payment_value
1430,e04abd8149ef81b95221e88f6ed9ab6a,1,boleto,1,349.01
28903,c1d4211b3dae76144deccd6c74144a88,1,boleto,1,54.51
39345,5cf925b116421afa85ee25e99b4c34fb,1,boleto,1,106.81
39368,51eb2eebd5d76a24625b31c33dd41449,1,boleto,1,77.06
40373,12a95a3c06dbaec84bcfb0e2da5d228a,1,boleto,1,95.76
41416,3c0b8706b065f9919d0505d3b3343881,1,boleto,1,157.19
44659,2eecb0d85f281280f79fa00f9cec1a95,1,boleto,1,154.23
44706,88083e8f64d95b932164187484d90212,1,boleto,1,127.04
60305,2babbb4b15e6d2dfe95e2de765c97bce,1,boleto,1,106.81
64205,7013bcfc1c97fe719a7b5e05e61c12db,1,boleto,1,65.52


In [17]:
# Analyze order statuses for orders with missing carrier delivery date
orders.loc[orders["order_delivered_carrier_date"].isna(), "order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [18]:
# Investigate delivered orders with missing carrier delivery date
delivered_missing_carrier = orders.loc[ 
    orders["order_delivered_carrier_date"].isna() & (orders["order_status"] == "delivered")]

delivered_missing_carrier

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


In [19]:
# Analyze invoiced orders with missing carrier delivery dates
orders.loc[orders["order_delivered_carrier_date"].isna() & (orders["order_status"] == "invoiced")].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaT,NaT,2018-08-21
455,38b7efdf33dd5561f4f5d4f6e07b0414,021e84751ba0ead75b6d314a6ead88d9,invoiced,2017-08-01 18:17:41,2017-08-01 18:32:30,NaT,NaT,2017-08-28
1833,51b0dccc8596ce37a930dff2d63a10a2,31bf1057b00f14804278590bbac18b1b,invoiced,2017-05-05 22:34:48,2017-05-05 22:45:12,NaT,NaT,2017-06-06
2116,5504eaa5a86eb25fa666cf2e6b96c701,a30be38e1ed0ffb39b318bf1d15e98d9,invoiced,2017-11-29 08:33:06,2017-11-29 08:56:23,NaT,NaT,2017-12-20


In [20]:
# check delivery timeline for invoiced orders with missing carrier delivery dates
invoiced_missing_carrier = orders.loc[
    orders["order_delivered_carrier_date"].isna() & (orders["order_status"] == "invoiced")]
invoiced_missing_carrier[[                                                               
    "order_status",                              
    "order_purchase_timestamp",       
    "order_approved_at",              
    "order_delivered_carrier_date",   
    "order_delivered_customer_date"]].isna().sum()

# Invoiced orders have purchase and approval timestamps, but delivery timestamps are missing because they have not reached the delivery stage.

order_status                       0
order_purchase_timestamp           0
order_approved_at                  0
order_delivered_carrier_date     314
order_delivered_customer_date    314
dtype: int64

In [21]:
# check delivery timeline for processing orders with missing carrier delivery dates
processing_missing_carrier = orders.loc[
    orders["order_delivered_carrier_date"].isna() & (orders["order_status"] == "processing")]

processing_missing_carrier[[                                                                
    "order_status",                              
    "order_purchase_timestamp",       
    "order_approved_at",              
    "order_delivered_carrier_date",   
    "order_delivered_customer_date"]].isna().sum()

order_status                       0
order_purchase_timestamp           0
order_approved_at                  0
order_delivered_carrier_date     301
order_delivered_customer_date    301
dtype: int64

In [22]:
# Investigate unavailable orders with missing carrier delivery dates
unavailable_missing_carrier = orders.loc[
    orders["order_delivered_carrier_date"].isna() & (orders["order_status"] == "unavailable")]

unavailable_missing_carrier[[                                                                
    "order_status",                              
    "order_purchase_timestamp",       
    "order_approved_at",              
    "order_delivered_carrier_date",   
    "order_delivered_customer_date"]].isna().sum()

order_status                       0
order_purchase_timestamp           0
order_approved_at                  0
order_delivered_carrier_date     609
order_delivered_customer_date    609
dtype: int64

In [23]:
# check created orders with missing carrier delivery dates
created_missing_carrier = orders.loc[
    orders["order_delivered_carrier_date"].isna() & (orders["order_status"] == "created")]

created_missing_carrier[[                                                                
    "order_status",                              
    "order_purchase_timestamp",       
    "order_approved_at",              
    "order_delivered_carrier_date",   
    "order_delivered_customer_date"]].isna().sum()

order_status                     0
order_purchase_timestamp         0
order_approved_at                5
order_delivered_carrier_date     5
order_delivered_customer_date    5
dtype: int64

In [24]:
# check approved orders with missing carrier delivery dates
approved_missing_carrier = orders.loc[
    orders["order_delivered_carrier_date"].isna() & (orders["order_status"] == "approved")]

approved_missing_carrier[[                                                                
    "order_status",                              
    "order_purchase_timestamp",       
    "order_approved_at",              
    "order_delivered_carrier_date",   
    "order_delivered_customer_date"]].isna().sum()

# Missing carrier delivery dates are retained as NaT. Most missing values are consistent with the order lifecycle,
# while a small number of delivered orders have incomplete timestamps.

order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     2
order_delivered_customer_date    2
dtype: int64

In [25]:
# Analyze order statuses for orders with missing customer delivery dates
orders.loc[orders["order_delivered_customer_date"].isna(), "order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [26]:
# Investigate delivered orders with missing customer delivery dates
delivered_missing_customer = orders.loc[
    orders["order_delivered_customer_date"].isna() & (orders["order_status"] == "delivered")]

delivered_missing_customer

# Missing customer delivery dates are retained as NaT. Delivered orders with missing timestemps are treated as incomplete records. 
# because exact delivery date can't be reliably reconstructed.

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [27]:
# Analyze review score for reviews with missing comment titles
reviews =pd.read_parquet(cleaned_folder/"olist_order_reviews_dataset.parquet")

reviews.loc[reviews["review_comment_title"].isna(), "review_score"].value_counts().sort_index()

review_score
1     9551
2     2673
3     7355
4    17407
5    50670
Name: count, dtype: int64

In [28]:
# check whether reviwe messages are available when titles are missing 
reviews.loc[reviews["review_comment_title"].isna(), "review_comment_message"].notna().value_counts()

# Missing review titles are retained as NaN because there is norelible basis to reconstruct the original title.

review_comment_message
False    56518
True     31138
Name: count, dtype: int64

In [29]:
# Analyze review scores for reviews with missing comment messages
reviews.loc[reviews["review_comment_message"].isna(), "review_score"].value_counts().sort_index()

review_score
1     2679
2     1006
3     4622
4    13166
5    36774
Name: count, dtype: int64

In [30]:
# check whether review titles are available when messages are missing
reviews.loc[reviews["review_comment_message"].isna(), "review_score"].notna().value_counts()

# Missing review massages are retained as NaN because customers may provide a title without a detailed massage.

review_score
True    58247
Name: count, dtype: int64

In [31]:
# load products datasets for missing value investigation
products = pd.read_parquet(cleaned_folder/"olist_products_dataset.parquet")

# Analyze products with missing category names
products.loc[products["product_category_name"].isna()].shape

(610, 9)

In [32]:
products.loc[products["product_category_name"].isna()].isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [33]:
# Inspecting products with missing category and product attributes 
products.loc[products["product_category_name"].isna()].head(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,<NA>,<NA>,<NA>,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,<NA>,<NA>,<NA>,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,<NA>,<NA>,<NA>,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,<NA>,<NA>,<NA>,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,<NA>,<NA>,<NA>,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,<NA>,<NA>,<NA>,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,<NA>,<NA>,<NA>,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,<NA>,<NA>,<NA>,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,<NA>,<NA>,<NA>,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,<NA>,<NA>,<NA>,1400.0,25.0,25.0,25.0


In [34]:
# check whether products with missing metadata are used in orders 
order_items = pd.read_parquet(cleaned_folder/"olist_order_items_dataset.parquet")

missing_category_products = products.loc[products["product_category_name"].isna(),"product_id"]

missing_category_products.isin(order_items["product_id"]).value_counts()

# Retain missing product metadata because these products are used in orders and the missing values cannot be reliably reconstructed from available data.

product_id
True    610
Name: count, dtype: int64

In [35]:
# Investigate products with missing physical diamensions
products.loc[
    products[[
        "product_weight_g",               
        "product_length_cm",               
        "product_height_cm",              
        "product_width_cm"]
    ].isna().any(axis=1)]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN


In [36]:
# check whether products with missing physical measurements are used in orders
missing_dimension_products = products.loc[
    products[[
        "product_weight_g",               
        "product_length_cm",               
        "product_height_cm",              
        "product_width_cm"]
    ].isna().any(axis=1),"product_id"]

missing_dimension_products.isin(order_items["product_id"]).value_counts()

# Product missing values are retained because the affected products are used in orders and missing attributes cannot be reliably reconstructed.

product_id
True    2
Name: count, dtype: int64

In [37]:
# Inspecting duplicate rows in geolocation dataset
geolocation = pd.read_parquet(cleaned_folder/"olist_geolocation_dataset.parquet")

geolocation[geolocation.duplicated(keep=False)].head(20)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
6,01047,-23.546273,-46.641225,sao paulo,SP
7,01013,-23.546923,-46.634264,sao paulo,SP
8,01029,-23.543769,-46.634278,sao paulo,SP
9,01011,-23.547640,-46.636032,sao paulo,SP
10,01013,-23.547325,-46.634184,sao paulo,SP
13,01012,-23.548946,-46.634671,sao paulo,SP
15,01046,-23.546081,-46.644820,sao paulo,SP


In [38]:
geolocation.duplicated().sum()

np.int64(261831)

In [39]:
geolocation.shape

(1000163, 5)

In [40]:
# Remove exact duplicate rows from geaolocation data 
geolocation_clean = geolocation.drop_duplicates()

In [41]:
# Veryfy duplicate removal
print("Before:",len(geolocation))
print("After:",len(geolocation_clean))
print("Duplicate Remaining:",geolocation_clean.duplicated().sum())

Before: 1000163
After: 738332
Duplicate Remaining: 0


In [42]:
# For parquet file
output_file = cleaned_folder/"Olist_geolocation_dataset.parquet"
geolocation_clean.to_parquet(output_file,index=False)

# for csv file
geolocation_clean.to_csv(cleaned_folder/"Olist_geolocation_dataset.csv",index=False)

print("Saved")

Saved


In [43]:
check_duplicate = pd.read_parquet(output_file).duplicated().sum()
check_duplicate

np.int64(0)

In [44]:
# Final missing value check
datasets = {
    "orders": orders,
    "reviews": reviews,
    "products" : products,
    "geolocation" : geolocation_clean
}

for name, df in datasets.items():
    print(f"\n{'-'*50}")
    print(name)
    print(df.isna().sum()[df.isna().sum() > 0])
    


--------------------------------------------------
orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--------------------------------------------------
reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

--------------------------------------------------
products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--------------------------------------------------
geolocation
Series([], dtype: int64)


In [45]:
# Final data quality summary
datasets = {
    "orders": orders,
    "reviews": reviews,
    "products" : products,
    "geolocation" : geolocation_clean
}

for name, df in datasets.items():
    print(f"\n{'-'*50}")
    print(name)
    print("Rows:",len(df))
    print("Columns:",len(df.columns))
    print("Missing values:",df.isna().sum().sum())
    print("Duplicate rows:",df.duplicated().sum())
    


--------------------------------------------------
orders
Rows: 99441
Columns: 8
Missing values: 4908
Duplicate rows: 0

--------------------------------------------------
reviews
Rows: 99224
Columns: 7
Missing values: 145903
Duplicate rows: 0

--------------------------------------------------
products
Rows: 32951
Columns: 9
Missing values: 2448
Duplicate rows: 0

--------------------------------------------------
geolocation
Rows: 738332
Columns: 5
Missing values: 0
Duplicate rows: 0


In [46]:
# Final validation of cleaned parquet datasets
parquet_files = list(cleaned_folder.glob("*.parquet"))

for file in parquet_files:
    df = pd.read_parquet(file)
    print(f"\n{'-'*50}")
    print(file.name)
    print("Rows:",len(df))
    print("Columns:",len(df.columns))
    print("Missing values:",df.isna().sum().sum())
    print("Duplicate rows:",df.duplicated().sum())
    


--------------------------------------------------
olist_customers_dataset.parquet
Rows: 99441
Columns: 5
Missing values: 0
Duplicate rows: 0

--------------------------------------------------
olist_geolocation_dataset.parquet
Rows: 738332
Columns: 5
Missing values: 0
Duplicate rows: 0

--------------------------------------------------
olist_orders_dataset.parquet
Rows: 99441
Columns: 8
Missing values: 4908
Duplicate rows: 0

--------------------------------------------------
olist_order_items_dataset.parquet
Rows: 112650
Columns: 7
Missing values: 0
Duplicate rows: 0

--------------------------------------------------
olist_order_payments_dataset.parquet
Rows: 103886
Columns: 5
Missing values: 0
Duplicate rows: 0

--------------------------------------------------
olist_order_reviews_dataset.parquet
Rows: 99224
Columns: 7
Missing values: 145903
Duplicate rows: 0

--------------------------------------------------
olist_products_dataset.parquet
Rows: 32951
Columns: 9
Missing values:

In [47]:
orders["order_id"].nunique()

99441

In [48]:
orders["customer_id"].nunique()

99441

In [49]:
orders["customer_id"].value_counts().value_counts().sort_index()

count
1    99441
Name: count, dtype: int64

In [50]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [52]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64